In [3]:
import json
from collections import defaultdict
from scipy.stats import pearsonr, spearmanr
from tqdm import tqdm

## Human-annotated faithfulness scores

In [22]:
human_annotation_file = "fact_check_sample.json" # path to original data with human annotations
model_faithfulness_scores = defaultdict(list)

In [23]:
# Dictionary to store faithfulness scores for each summary (doc_id)
human_faithfulness_scores = {}
# Dictionary to store faithfulness scores per model
model_faithfulness_scores = defaultdict(list)

In [24]:
def compute_human_faithfulness_score(factuality_labels):
    total_sentences = len(factuality_labels)
    num_errors = sum(factuality_labels)
    return 1 - (num_errors / total_sentences)

In [ ]:
with open(human_annotation_file, "r") as f:
    for line in tqdm(f, desc="Processing human annotations"):
        data = json.loads(line.strip())  # Load each JSON object
        doc_id = data["doc_id"]
        factuality_labels = data["raw_annotations"]["annotator_0"]["factuality_labels"]

        # Compute faithfulness score
        faithfulness_score = compute_human_faithfulness_score(factuality_labels)
        human_faithfulness_scores[doc_id] = faithfulness_score


In [ ]:
print("\nHuman Annotated Faithfulness Scores:")
for doc_id, score in human_faithfulness_scores.items():
    print(f"Doc ID: {doc_id}, Faithfulness Score: {score:.4f}")


In [ ]:
# Save to a JSON file
output_file = "human_faithfulness_scores.json"
with open(output_file, "w") as f:
    json.dump(human_faithfulness_scores, f, indent=4)

print(f"\nHuman faithfulness scores saved to {output_file}")

In [ ]:
# Model-Level human faithfulness scores
with open(human_annotation_file, "r") as f:
    for line in tqdm(f, desc="Processing human annotations"):
        data = json.loads(line.strip())  # Load each JSON object
        model = data["model"]
        factuality_labels = data["raw_annotations"]["annotator_0"]["factuality_labels"]

        # Compute faithfulness score
        faithfulness_score = compute_human_faithfulness_score(factuality_labels)
        model_faithfulness_scores[model].append(faithfulness_score)

# Compute model-level average faithfulness scores
human_system_scores = {model: sum(scores) / len(scores) for model, scores in model_faithfulness_scores.items()}

print("\nModel-Level Human Faithfulness Scores:")
for model, score in human_system_scores.items():
    print(f"Model: {model}, Average Faithfulness Score: {score:.4f}")


In [ ]:
# Save to a JSON file
output_file = "human_system_scores.json"
with open(output_file, "w") as f:
    json.dump(human_system_scores, f, indent=4)

print(f"\nHuman system faithfulness scores saved to {output_file}")

## Performance of faithfulness evaluation of FineSure

In [ ]:
# FineSurE system-level scores
finesure_system_scores = {
    "bart": 62.7,
    "gpt3.5": 69.4,
    "gpt4": 75.0
}

# Human system-level scores (previously computed)
human_system_scores = {
    "bart": 0.5317460317460317,
    "gpt3.5": 0.888888888888889,
    "gpt4": 0.75
}

# Compute Pearson correlation (system-level)
human_values = [human_system_scores[m] for m in human_system_scores.keys()]
finesure_values = [finesure_system_scores[m] for m in human_system_scores.keys()]

pearson_corr_system, _ = pearsonr(human_values, finesure_values)

# Compute Spearman rank correlation (system-level)
human_ranking = sorted(human_system_scores.keys(), key=lambda x: human_system_scores[x], reverse=True)
finesure_ranking = sorted(finesure_system_scores.keys(), key=lambda x: finesure_system_scores[x], reverse=True)

human_ranking_scores = [human_ranking.index(m) + 1 for m in human_system_scores.keys()]
finesure_ranking_scores = [finesure_ranking.index(m) + 1 for m in human_system_scores.keys()]

spearman_corr_system, _ = spearmanr(human_ranking_scores, finesure_ranking_scores)

# Output results
print("\nSummary-level Pearson Correlation (FineSurE vs Human Annotations):")
print(f"Pearson Corr: {pearson_corr_system:.4f}")

print("\nSummary-level Rank Correlation (FineSurE vs Human Ranking):")
print(f"Spearman Corr: {spearman_corr_system:.4f}")

print("\nSystem-level Rank Correlation (FineSurE vs Human Ranking):")
print(f"Spearman Corr: {spearman_corr_system:.4f}")

## Performance of faithfulness evaluation of ROUGE

In [ ]:
import json
from scipy.stats import pearsonr, spearmanr
from collections import defaultdict
from tqdm import tqdm

# Define file paths
rouge_file_path = "rouge_faithfulness.json"
human_annotation_file = "human_faithfulness_scores.json"

# Load ROUGE results from JSON
with open(rouge_file_path, "r") as f:
    rouge_data = json.load(f)

# Load human faithfulness scores from JSON
with open(human_annotation_file, "r") as f:
    human_faithfulness_scores = json.load(f)

# Prepare summary-level ROUGE scores and human scores
summary_human_scores = []
summary_rouge_scores = {"rouge1": [], "rouge2": [], "rougeL": []}

for entry in tqdm(rouge_data, desc="Processing ROUGE summary scores"):
    doc_id = entry["doc_id"]
    
    if doc_id in human_faithfulness_scores:
        summary_human_scores.append(human_faithfulness_scores[doc_id])
        summary_rouge_scores["rouge1"].append(entry["rouge1"])
        summary_rouge_scores["rouge2"].append(entry["rouge2"])
        summary_rouge_scores["rougeL"].append(entry["rougeL"])

# Compute Pearson and Spearman correlations for summary-level
pearson_results_summary = {}
spearman_results_summary = {}

for rouge_type in ["rouge1", "rouge2", "rougeL"]:
    pearson_corr, _ = pearsonr(summary_human_scores, summary_rouge_scores[rouge_type])
    spearman_corr, _ = spearmanr(summary_human_scores, summary_rouge_scores[rouge_type])
    
    pearson_results_summary[rouge_type] = pearson_corr
    spearman_results_summary[rouge_type] = spearman_corr

# Compute system-level rank correlation
system_human_scores = defaultdict(list)
system_rouge_scores = defaultdict(lambda: {"rouge1": [], "rouge2": [], "rougeL": []})

# Group scores by system
for entry in tqdm(rouge_data, desc="Processing ROUGE system scores"):
    model = entry["model"]
    doc_id = entry["doc_id"]
    
    if doc_id in human_faithfulness_scores:
        system_human_scores[model].append(human_faithfulness_scores[doc_id])
        system_rouge_scores[model]["rouge1"].append(entry["rouge1"])
        system_rouge_scores[model]["rouge2"].append(entry["rouge2"])
        system_rouge_scores[model]["rougeL"].append(entry["rougeL"])

# Compute average system-level scores
avg_human_system_scores = {model: sum(scores) / len(scores) for model, scores in system_human_scores.items()}
avg_rouge_system_scores = {model: {
    "rouge1_avg": sum(scores["rouge1"]) / len(scores["rouge1"]),
    "rouge2_avg": sum(scores["rouge2"]) / len(scores["rouge2"]),
    "rougeL_avg": sum(scores["rougeL"]) / len(scores["rougeL"])
} for model, scores in system_rouge_scores.items()}

# Compute system-level rank correlation
spearman_results_system = {}

for rouge_type in ["rouge1_avg", "rouge2_avg", "rougeL_avg"]:
    human_ranking = sorted(avg_human_system_scores.keys(), key=lambda x: avg_human_system_scores[x], reverse=True)
    method_ranking = sorted(avg_rouge_system_scores.keys(), key=lambda x: avg_rouge_system_scores[x][rouge_type], reverse=True)

    human_ranking_scores = [human_ranking.index(m) + 1 for m in avg_human_system_scores.keys()]
    method_ranking_scores = [method_ranking.index(m) + 1 for m in avg_human_system_scores.keys()]

    spearman_corr, _ = spearmanr(human_ranking_scores, method_ranking_scores)
    spearman_results_system[rouge_type] = spearman_corr

# Output results
print("\nSummary-level Pearson Correlation:")
for rouge_type, corr in pearson_results_summary.items():
    print(f"{rouge_type}: {corr:.4f}")

print("\nSummary-level Spearman Correlation:")
for rouge_type, corr in spearman_results_summary.items():
    print(f"{rouge_type}: {corr:.4f}")

print("\nSystem-level Rank Correlation:")
for rouge_type, corr in spearman_results_system.items():
    print(f"{rouge_type}: {corr:.4f}")


## Performance of faithfulness evaluation of BERTScore

In [ ]:
# Define file paths
bertscore_file_path = "bertscore_faithfulness.json"
human_annotation_file = "human_faithfulness_scores.json"

# Load BERTScore results from JSON
with open(bertscore_file_path, "r") as f:
    bertscore_data = json.load(f)

# Load human faithfulness scores from JSON
with open(human_annotation_file, "r") as f:
    human_faithfulness_scores = json.load(f)

# Prepare summary-level BERTScore and human scores
summary_human_scores = []
summary_bertscore_f1 = []

# Match BERTScore F1 scores with human annotations
for entry in bertscore_data:
    doc_id = entry["doc_id"]
    if doc_id in human_faithfulness_scores:
        summary_human_scores.append(human_faithfulness_scores[doc_id])
        summary_bertscore_f1.append(entry["BERTScore_F1"])

# Compute Pearson and Spearman correlations for summary-level evaluation
pearson_corr_summary, _ = pearsonr(summary_human_scores, summary_bertscore_f1)
spearman_corr_summary, _ = spearmanr(summary_human_scores, summary_bertscore_f1)

# Compute system-level rank correlation
system_human_scores = defaultdict(list)
system_bertscore_f1 = defaultdict(list)

# Group scores by model
for entry in bertscore_data:
    model = entry["model"]
    doc_id = entry["doc_id"]
    if doc_id in human_faithfulness_scores:
        system_human_scores[model].append(human_faithfulness_scores[doc_id])
        system_bertscore_f1[model].append(entry["BERTScore_F1"])

# Compute average system-level scores
avg_human_system_scores = {model: sum(scores) / len(scores) for model, scores in system_human_scores.items()}
avg_bertscore_system_scores = {model: sum(scores) / len(scores) for model, scores in system_bertscore_f1.items()}

# Compute system-level rank correlation
human_ranking = sorted(avg_human_system_scores.keys(), key=lambda x: avg_human_system_scores[x], reverse=True)
bertscore_ranking = sorted(avg_bertscore_system_scores.keys(), key=lambda x: avg_bertscore_system_scores[x], reverse=True)

human_ranking_scores = [human_ranking.index(m) + 1 for m in avg_human_system_scores.keys()]
bertscore_ranking_scores = [bertscore_ranking.index(m) + 1 for m in avg_human_system_scores.keys()]

spearman_corr_system, _ = spearmanr(human_ranking_scores, bertscore_ranking_scores)

# Output results
print("\nSummary-level Pearson Correlation (BERTScore-F1 vs Human Annotations):")
print(f"Pearson Corr: {pearson_corr_summary:.4f}")

print("\nSummary-level Spearman Correlation (BERTScore-F1 vs Human Annotations):")
print(f"Spearman Corr: {spearman_corr_summary:.4f}")

print("\nSystem-level Rank Correlation (BERTScore-F1 vs Human Ranking):")
print(f"Spearman Corr: {spearman_corr_system:.4f}")